# Week 4 Tue: Confidence intervals and hypothesis testing intuition

Estimated time: 8 hours

## Why this matters
Confidence intervals and hypothesis testing intuition is part of real quant work inside math rebuild ii: statistics, inference, regression, optimization, and portfolio theory research, trading, or risk workflows.

## Session plan
- Session 1 (45 min): Confidence interval intuition.
- Session 2 (55 min): Hypothesis testing as evidence check.
- Session 3 (55 min): Common misuse in finance.
- Session 4 (55 min): Code interval examples.
- Session 5 (30 min): Interview recap.

## Concept notes
### Confidence interval intuition
A confidence interval gives a plausible range for an unknown quantity based on the sample.

Technical note: It reflects sampling uncertainty around an estimator under model assumptions.

Finance example: A strategy with a positive sample mean may still have a wide interval that includes weak or negative true edge.

### Hypothesis testing
A test asks whether the evidence is strong enough to challenge a baseline claim.

Technical note: In many settings the baseline claim is a null hypothesis such as zero mean effect.

Finance example: You may test whether a signal's average return differs meaningfully from zero.

### Evidence versus certainty
A test result is evidence, not proof.

Technical note: Statistical significance does not automatically imply economic significance or robustness.

Finance example: A tiny effect can be statistically noticeable in a large sample yet useless after costs.

## Continuity prompt
- What from yesterday's topic still needs reinforcement?
- What should tomorrow build on from today?

## Real-world dataset for today's labs
Use `curriculum/datasets/real_market_prices.csv` as your default market panel (SPY, QQQ, TLT, GLD).


## Code Lab 1: Rough confidence interval

In [ ]:
import numpy as np

sample = np.array([0.01, 0.02, -0.01, 0.00, 0.03, 0.01, -0.02, 0.02])
mean = sample.mean()
std = sample.std(ddof=1)
se = std / np.sqrt(len(sample))
ci_low = mean - 1.96 * se
ci_high = mean + 1.96 * se
print(round(mean, 4), round(ci_low, 4), round(ci_high, 4))


## Practice recap
- Why is statistical significance not enough in finance?
  Suggested answer: Because trading viability also depends on effect size, costs, risk, stability, and implementation realism.
- What does a wide confidence interval suggest?
  Suggested answer: It suggests high uncertainty about the true parameter given the sample and assumptions.

## Interview drill
- Q: How would you explain a confidence interval simply?
  A: It is a range of plausible values for the unknown quantity based on what the sample tells us.

## 4+ Hour Completion Roadmap

Use this minimum structure to turn today's notebook into a serious study session:

- Phase 1 (60 min): Read concept notes and rewrite the core idea from memory.
- Phase 2 (70 min): Complete and extend all code labs with at least one variation.
- Phase 3 (65 min): Do formula retrieval, scenario analysis, and error-log updates.
- Phase 4 (65 min): Write a mini memo and practice interview responses aloud.

## Formula Rewrite Drill

In [ ]:
import pandas as pd

formula_drill = pd.DataFrame(
    [
        {"formula": "E[X] = sum_i p_i x_i", "from_memory": "", "term_explanations": ""},
        {"formula": "Var(X) = E[(X - E[X])^2]", "from_memory": "", "term_explanations": ""},
        {"formula": "W_t = W_0 * product(1 + r_t)", "from_memory": "", "term_explanations": ""},
        {"formula": "Topic formula for: Confidence intervals and hypothesis testing intuition", "from_memory": "", "term_explanations": ""},
    ]
)
print(formula_drill)


## Real Market Data Lab (Useful From Day 1)

This section uses a local CSV snapshot of real market prices so the notebook remains reproducible.

Dataset: `curriculum/datasets/real_market_prices.csv`
Symbols: SPY, QQQ, TLT, GLD

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

data_path = Path("curriculum/datasets/real_market_prices.csv")
market = pd.read_csv(data_path, parse_dates=["date"]).sort_values(["date", "symbol"])

print("Rows:", len(market))
print("Date range:", market["date"].min().date(), "to", market["date"].max().date())
print("Symbols:", sorted(market["symbol"].unique()))
print(market.head())

prices = market.pivot(index="date", columns="symbol", values="close").dropna()
returns = prices.pct_change().dropna()

summary = pd.DataFrame(
    {
        "ann_return": returns.mean() * 252,
        "ann_vol": returns.std() * (252 ** 0.5),
        "max_drawdown": ((1 + returns).cumprod().div((1 + returns).cumprod().cummax()) - 1).min(),
    }
).sort_index()
print("\nRisk/return summary:")
print(summary.round(4))

cum = (1 + returns).cumprod()
cum.plot(figsize=(10, 5), title="Cumulative growth (base 1.0)")
plt.ylabel("Growth of $1")
plt.tight_layout()
plt.show()


## Real-World Takeaway Prompt

Write 5-8 lines for today's topic (Confidence intervals and hypothesis testing intuition):

1. Which symbol looked most volatile and why that matters.
2. Which pair looked most diversifying.
3. One realistic portfolio/risk decision you could make from this table.

## Interview Question + Python Solution Drill

Use this format for interview preparation:

1. State the question clearly.
2. Explain your approach in plain language.
3. Write Python code on real market data.
4. Interpret one risk caveat in words.

**Suggested data-source ladder**
- Source 1: yfinance pull (fresh market data)
- Source 2: Robinhood-style CSV export (if available locally)
- Source 3: local snapshot `curriculum/datasets/real_market_prices.csv` (reproducible fallback)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

prices = None
source_used = ""
USE_YFINANCE = False  # set True when you want a live market pull

try:
    import yfinance as yf
except ImportError:
    yf = None

if USE_YFINANCE and yf is not None:
    downloaded = yf.download(
        ["SPY", "QQQ", "TLT", "GLD"],
        period="2y",
        interval="1d",
        auto_adjust=True,
        progress=False,
    )
    if not downloaded.empty:
        if isinstance(downloaded.columns, pd.MultiIndex):
            if "Close" in downloaded.columns.get_level_values(0):
                prices = downloaded["Close"].copy()
            elif "Adj Close" in downloaded.columns.get_level_values(0):
                prices = downloaded["Adj Close"].copy()
        else:
            prices = downloaded.rename(columns=str.upper)
        source_used = "yfinance"

robinhood_export = Path("curriculum/datasets/robinhood_export.csv")
if prices is None and robinhood_export.exists():
    rh = pd.read_csv(robinhood_export, parse_dates=["date"])
    prices = rh.pivot(index="date", columns="symbol", values="close").sort_index()
    source_used = "robinhood_export_csv"

if prices is None:
    local = pd.read_csv(
        Path("curriculum/datasets/real_market_prices.csv"),
        parse_dates=["date"],
    )
    prices = local.pivot(index="date", columns="symbol", values="close").sort_index()
    source_used = "local_snapshot_csv"

prices = prices.dropna(how="all").ffill().dropna()
returns = prices.pct_change().dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

annualized_return = returns.mean() * 252
annualized_vol = returns.std() * np.sqrt(252)
sharpe_proxy = (annualized_return - 0.03) / annualized_vol

print("Source used:", source_used)
print("\nAnnualized return:")
print(annualized_return.round(4))
print("\nAnnualized volatility:")
print(annualized_vol.round(4))
print("\nSharpe proxy (rf=3%):")
print(sharpe_proxy.round(3))
print("\nRecent log returns:")
print(log_returns.tail().round(4))


## Scenario Analysis Drill

In [ ]:
import pandas as pd

scenarios = pd.DataFrame(
    [
        {"scenario": "base_case", "assumption": "", "expected_effect": ""},
        {"scenario": "bull_case", "assumption": "", "expected_effect": ""},
        {"scenario": "stress_case", "assumption": "", "expected_effect": ""},
    ]
)
print("Topic:", 'Confidence intervals and hypothesis testing intuition')
print(scenarios)


## Closed-Book Retrieval and Error Log

In [ ]:
import pandas as pd

retrieval_scorecard = pd.DataFrame(
    [
        {"prompt": "Explain today's concept in 3 lines", "score_0_to_5": None, "notes": ""},
        {"prompt": "Write one key formula without notes", "score_0_to_5": None, "notes": ""},
        {"prompt": "Give one realistic failure mode", "score_0_to_5": None, "notes": ""},
        {"prompt": "Connect to one trading/risk decision", "score_0_to_5": None, "notes": ""},
    ]
)

error_log = pd.DataFrame(
    [
        {
            "concept": "",
            "mistake": "",
            "correction": "",
            "next_review_date": "",
        }
    ]
)
print("Retrieval scorecard template:")
print(retrieval_scorecard)
print("\nError log template:")
print(error_log)


## Final 30-Minute Deliverable

Write a short memo (150-250 words) with this structure:

1. Core idea of Confidence intervals and hypothesis testing intuition in plain language.
2. One technical detail or formula and why it matters.
3. One practical quant use case.
4. One limitation or failure mode and how you would detect it.